<a href="https://colab.research.google.com/github/cbonnin88/LuminaStream/blob/main/data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
from datetime import datetime, timedelta
import random
import uuid
import json

# **Configuration**

In [2]:
num_users = 25000
start_date = datetime(2026,4,1)

In [5]:
plans = {'Basic':0.00,'Extra':5.99,'Premium':11.99,'Ultra':25.99}
plan_options = list(plans.keys())
plan_weights = [0.2,0.3,0.3,0.2]

countries = ['France','Germany','The Netherlands','United Kingdom','Belgium','Switzerland','Austria','Denmark','Norway','Spain','Ireland']

genders = ['Male','Female','Non-Binary']
gender_weights = [0.48,0.48,0.04]

devices = ['Mobile','Web','Smart Screen','Tablet']

event_types = ['session_start', 'search_content', 'view_channel', 'add_to_favorites', 'start_recording', 'app_crash', 'quality_change_auto', 'subscription_upgrade']
event_weight = [0.15, 0.2, 0.35, 0.1, 0.1, 0.01, 0.05, 0.04]

print(f'Generating data for {num_users} users....')

Generating data for 25000 users....


# **Generate and Write Users**

In [6]:
with open('lumina_users_scaled.json', 'w') as f:
    for i in range(num_users):
        user_id = f"LS_{i}"
        plan = random.choices(plan_options, weights=plan_weights)[0]
        user = {
            "user_id": user_id,
            "gender": random.choices(genders, weights=gender_weights)[0],
            "country": random.choice(countries),
            "signup_date": (start_date + timedelta(days=random.randint(0, 28))).isoformat(),
            "plan_type": plan,
            "subscription_price": plans[plan]
        }
        f.write(json.dumps(user) + '\n')

# **Generate and Write Events (Estimated around 300k events)**

In [8]:
print("Generating event stream (this may take a minute)...")
with open('lumina_events_scaled.json', 'w') as f:
    for i in range(num_users):
        user_id = f"LS_{i}"
        signup_dt = start_date + timedelta(days=random.randint(0, 28))

        # Each user has 1-4 distinct sessions
        for _ in range(random.randint(1, 4)):
            device_id = str(uuid.uuid4())[:8]
            device_type = random.choice(devices)

            # Each session has 5-15 events
            for _ in range(random.randint(5, 15)):
                ts = signup_dt + timedelta(hours=random.randint(1, 500))
                event = {
                    "user_id": user_id,
                    "device_id": device_id,
                    "device_type": device_type,
                    "event_type": random.choices(event_types, weights=event_weight)[0],
                    "timestamp": ts.isoformat(),
                    "content_id": random.choice(["TF1", "M6", "BBC One", "ZDF", "Sky Sport", "Canal+", "Arte", "Netflix Partner", None]),
                    "session_id": str(uuid.uuid4())[:12]
                }
                f.write(json.dumps(event) + '\n')

print("Done! Files 'lumina_users_scaled.json' and 'lumina_events_scaled.json' are ready.")

Generating event stream (this may take a minute)...
Done! Files 'lumina_users_scaled.json' and 'lumina_events_scaled.json' are ready.


In [10]:
display(user)

{'user_id': 'LS_24999',
 'gender': 'Male',
 'country': 'Norway',
 'signup_date': '2026-04-16T00:00:00',
 'plan_type': 'Ultra',
 'subscription_price': 25.99}

In [11]:
display(event)

{'user_id': 'LS_24999',
 'device_id': '094839ff',
 'device_type': 'Mobile',
 'event_type': 'quality_change_auto',
 'timestamp': '2026-04-26T00:00:00',
 'content_id': 'Netflix Partner',
 'session_id': 'cd72a411-4a6'}